# Monitorização da pressão assistencial nas urgências hospitalares do SNS

## Projeto 1 — Análise exploratória e diagnóstico operacional

Este notebook faz parte de um projeto de portefólio em Health Data Science focado em serviços de urgência, tempos de espera, fluxo de doentes e congestionamento hospitalar.

O objetivo deste primeiro projeto é caracterizar e monitorizar a pressão assistencial nas urgências hospitalares do SNS em Portugal, usando dados públicos agregados.

## Pergunta principal

Como evolui a pressão assistencial nos serviços de urgência hospitalar do SNS em Portugal e que indicadores públicos permitem monitorizar padrões de congestionamento ao longo do tempo e entre instituições?

## Objetivos

### Objetivo geral

Caracterizar e monitorizar a pressão assistencial nas urgências hospitalares do SNS em Portugal, usando dados públicos sobre volume de atendimentos, tempos médios de espera, triagem e internamento.

### Objetivos específicos

- Analisar a evolução temporal do volume de atendimentos urgentes.
- Analisar a evolução do tempo médio entre triagem e primeira observação médica.
- Explorar a relação entre volume, tempo de espera, triagem e taxa de internamento.
- Comparar padrões entre instituições hospitalares e/ou regiões.
- Identificar indicadores úteis para monitorização de congestionamento.
- Construir um indicador exploratório de pressão assistencial.

## Estrutura prevista do notebook

1. Introdução e contexto
2. Pergunta principal e objetivos
3. Fontes de dados
4. Compreensão inicial dos dados
5. Qualidade dos dados
6. Preparação das variáveis
7. Análise exploratória
8. Indicador exploratório de pressão assistencial
9. Síntese dos principais achados
10. Limitações
11. Próximos passos

## 1. Importação de bibliotecas e configuração inicial

Nesta secção serão carregadas as bibliotecas necessárias para a análise exploratória e será definida a localização dos ficheiros de dados.

O objetivo inicial é apenas confirmar que o dataset principal foi guardado corretamente e que pode ser lido no ambiente de trabalho.

In [1]:
from pathlib import Path

import pandas as pd

In [5]:
DATA_RAW = Path("../data/raw")
main_dataset_path = DATA_RAW / "monitorizacao_sazonal_csh_raw.csv"

main_dataset_path.exists()

True

In [7]:
df_main = pd.read_csv(main_dataset_path, sep=";")

df_main.head()

,Período,Região/ARS,Indicador,Valor,Unidade,ID
0,2017-02-13,ARS Lisboa e Vale do Tejo,Tempo médio de espera entre a triagem e a prim...,47.867240,minuto,2017-02-13/ARS Lisboa e Vale do Tejo/Tempo méd...
1,2017-02-14,Portugal Continental,Tempo médio de espera entre a triagem e a prim...,46.503490,minuto,2017-02-14/Portugal Continental/Tempo médio de...
2,2017-02-14,ARS Alentejo,Número estimado de episódios de urgência,1048.000000,episódio urgência,2017-02-14/ARS Alentejo/Número estimado de epi...
3,2017-02-14,ARS Alentejo,Taxa diária de atendimentos urgentes com prior...,39.185390,epis. urg./100.000 resid.,2017-02-14/ARS Alentejo/Taxa diária de episódi...
4,2017-02-14,ARS Norte,Taxa diária de atendimentos urgentes com inter...,8.190686,%,2017-02-14/ARS Norte/Taxa diária de episódios ...


### Nota sobre leitura do ficheiro

O ficheiro CSV foi exportado do Portal da Transparência do SNS com separador `;`.

Por isso, a leitura com `pandas` deve usar o argumento `sep=";"`.

## 2. Compreensão inicial do dataset principal

Nesta secção é feita uma primeira inspeção ao dataset principal, incluindo dimensão da tabela, nomes das colunas, tipos de dados e indicadores disponíveis.

O objetivo é perceber a estrutura dos dados antes de qualquer transformação.

In [10]:
df_main.shape

(82154, 6)

In [11]:
df_main.columns.tolist()

['Período', 'Região/ARS', 'Indicador', 'Valor', 'Unidade', 'ID']

In [12]:
df_main.info()

<class 'pandas.DataFrame'>
RangeIndex: 82154 entries, 0 to 82153
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Período     82154 non-null  str    
 1   Região/ARS  82154 non-null  str    
 2   Indicador   82154 non-null  str    
 3   Valor       82154 non-null  float64
 4   Unidade     82154 non-null  str    
 5   ID          82154 non-null  str    
dtypes: float64(1), str(5)
memory usage: 3.8 MB


In [13]:
df_main.head(10)

,Período,Região/ARS,Indicador,Valor,Unidade,ID
0,2017-02-13,ARS Lisboa e Vale do Tejo,Tempo médio de espera entre a triagem e a prim...,47.867240,minuto,2017-02-13/ARS Lisboa e Vale do Tejo/Tempo méd...
1,2017-02-14,Portugal Continental,Tempo médio de espera entre a triagem e a prim...,46.503490,minuto,2017-02-14/Portugal Continental/Tempo médio de...
2,2017-02-14,ARS Alentejo,Número estimado de episódios de urgência,1048.000000,episódio urgência,2017-02-14/ARS Alentejo/Número estimado de epi...
3,2017-02-14,ARS Alentejo,Taxa diária de atendimentos urgentes com prior...,39.185390,epis. urg./100.000 resid.,2017-02-14/ARS Alentejo/Taxa diária de episódi...
4,2017-02-14,ARS Norte,Taxa diária de atendimentos urgentes com inter...,8.190686,%,2017-02-14/ARS Norte/Taxa diária de episódios ...
5,2017-02-14,ARS Norte,Tempo médio de espera entre a triagem e a prim...,41.273070,minuto,2017-02-14/ARS Norte/Tempo médio de espera ent...
6,2017-02-15,Portugal Continental,Taxa diária de atendimentos urgentes com prior...,38.654630,epis. urg./100.000 resid.,2017-02-15/Portugal Continental/Taxa diária de...
7,2017-02-15,ARS Lisboa e Vale do Tejo,Taxa diária de atendimentos urgentes com inter...,8.884231,%,2017-02-15/ARS Lisboa e Vale do Tejo/Taxa diár...
8,2017-02-15,ARS Lisboa e Vale do Tejo,Tempo médio de espera entre a triagem e a prim...,57.487580,minuto,2017-02-15/ARS Lisboa e Vale do Tejo/Tempo méd...
9,2017-02-15,ARS Algarve,Taxa diária de atendimentos urgentes com inter...,6.781915,%,2017-02-15/ARS Algarve/Taxa diária de episódio...


In [14]:
df_main["Indicador"].value_counts()

Indicador
Número estimado de episódios de urgência                                                              20838
Taxa diária de atendimentos urgentes com prioridade verde ou azul                                     20541
Taxa diária de atendimentos urgentes com internamento                                                 20540
Tempo médio de espera entre a triagem e a primeira observação médica (rede de urgência hospitalar)    20235
Name: count, dtype: int64

In [ ]:
print(f"Período mínimo: {df_main['Período'].min()}")
print(f"Período máximo: {df_main['Período'].max()}")
print(f"Regiões: {df_main['Região/ARS'].value_counts()}")

Período mínimo: 2016-11-01
Período máximo: 2026-05-09
Regiôes: Região/ARS
ARS Lisboa e Vale do Tejo    13892
Portugal Continental         13892
ARS Alentejo                 13892
ARS Norte                    13892
ARS Centro                   13892
ARS Algarve                  12694
Name: count, dtype: int64


In [33]:
# Período min e máximo por região/ars
df_main.groupby('Região/ARS')['Período'].agg(['min', 'max'])


,min,max
Região/ARS,,
ARS Alentejo,2016-11-01,2026-05-09
ARS Algarve,2016-11-01,2026-05-09
ARS Centro,2016-11-01,2026-05-09
ARS Lisboa e Vale do Tejo,2016-11-01,2026-05-09
ARS Norte,2016-11-01,2026-05-09
Portugal Continental,2016-11-01,2026-05-09


In [34]:
tabela_regiao_indicador = pd.crosstab(df_main['Região/ARS'], df_main['Indicador'])
tabela_regiao_indicador

Indicador,Número estimado de episódios de urgência,Taxa diária de atendimentos urgentes com internamento,Taxa diária de atendimentos urgentes com prioridade verde ou azul,Tempo médio de espera entre a triagem e a primeira observação médica (rede de urgência hospitalar)
Região/ARS,,,,
ARS Alentejo,3473,3473,3473,3473
ARS Algarve,3473,3175,3176,2870
ARS Centro,3473,3473,3473,3473
ARS Lisboa e Vale do Tejo,3473,3473,3473,3473
ARS Norte,3473,3473,3473,3473
Portugal Continental,3473,3473,3473,3473


### Nota sobre cobertura dos dados por região e indicador

A análise do número de registos por Região/ARS e por Indicador mostra que a ARS Algarve apresenta menor cobertura em alguns indicadores, especialmente no indicador "Tempo médio de espera entre a triagem e a primeira observação médica".

Esta diferença não aparece como valores nulos nas colunas, mas sim como menor número de registos disponíveis para essa combinação de região e indicador.

Por este motivo, comparações envolvendo a ARS Algarve, sobretudo no indicador de tempo médio de espera, devem ser interpretadas com cautela.

In [44]:
df_quality = df_main.copy()

df_quality["Período"] = pd.to_datetime(df_quality["Período"])

In [47]:
indicador_espera = "Tempo médio de espera"

df_espera = df_quality[df_quality["Indicador"].str.contains(indicador_espera, case=False, na=False)].copy()

df_espera["Indicador"].unique()

<StringArray>
['Tempo médio de espera entre a triagem e a primeira observação médica (rede de urgência hospitalar)']
Length: 1, dtype: str

In [48]:
df_espera.groupby("Região/ARS")["Período"].agg(['min', 'max', "count"])

,min,max,count
Região/ARS,,,
ARS Alentejo,2016-11-01,2026-05-09,3473
ARS Algarve,2016-11-01,2026-05-09,2870
ARS Centro,2016-11-01,2026-05-09,3473
ARS Lisboa e Vale do Tejo,2016-11-01,2026-05-09,3473
ARS Norte,2016-11-01,2026-05-09,3473
Portugal Continental,2016-11-01,2026-05-09,3473


In [51]:
datas_todas_regiões = set(df_espera["Período"].unique())

datas_alg = set(df_espera.loc[df_espera["Região/ARS"] == "ARS Algarve", "Período"].unique())

datas_em_falta_alg = sorted(datas_todas_regiões - datas_alg)

len(datas_em_falta_alg)

603

In [52]:
datas_em_falta_alg[:10], datas_em_falta_alg[-10:]

([Timestamp('2016-12-05 00:00:00'),
  Timestamp('2016-12-06 00:00:00'),
  Timestamp('2016-12-07 00:00:00'),
  Timestamp('2016-12-08 00:00:00'),
  Timestamp('2017-02-04 00:00:00'),
  Timestamp('2017-02-05 00:00:00'),
  Timestamp('2017-02-06 00:00:00'),
  Timestamp('2017-02-07 00:00:00'),
  Timestamp('2017-02-08 00:00:00'),
  Timestamp('2017-02-09 00:00:00')],
 [Timestamp('2026-04-22 00:00:00'),
  Timestamp('2026-04-23 00:00:00'),
  Timestamp('2026-04-24 00:00:00'),
  Timestamp('2026-04-25 00:00:00'),
  Timestamp('2026-04-26 00:00:00'),
  Timestamp('2026-04-27 00:00:00'),
  Timestamp('2026-04-28 00:00:00'),
  Timestamp('2026-05-03 00:00:00'),
  Timestamp('2026-05-05 00:00:00'),
  Timestamp('2026-05-06 00:00:00')])

In [54]:
df_faltas_algarve = pd.DataFrame({
    "data_em_falta": datas_em_falta_alg
})

df_faltas_algarve["ano"] = df_faltas_algarve["data_em_falta"].dt.year
df_faltas_algarve["mes"] = df_faltas_algarve["data_em_falta"].dt.month

df_faltas_algarve.groupby("ano").size()

ano
2016      4
2017    153
2018      8
2019    147
2023      4
2024    100
2025     66
2026    121
dtype: int64

In [55]:
df_faltas_algarve.groupby(["ano", "mes"]).size()

ano   mes
2016  12      4
2017  2      19
      3      29
      4      19
      5      10
      6      21
      7      26
      8      11
      10      1
      11      9
      12      8
2018  1       1
      2       1
      3       2
      10      1
      11      3
2019  5       2
      6       3
      7      26
      8      30
      9      25
      10     28
      11     24
      12      9
2023  1       3
      3       1
2024  4       1
      6      25
      7      30
      8      16
      9       5
      10      3
      11     10
      12     10
2025  1       2
      2       3
      3       1
      4       5
      5       6
      6       9
      7       1
      8       2
      9       3
      10      3
      11      7
      12     24
2026  1      31
      2      28
      3      31
      4      28
      5       3
dtype: int64

### Cobertura temporal do indicador de tempo médio de espera

Na análise de qualidade dos dados, verificou-se que a ARS Algarve apresenta menor cobertura no indicador "Tempo médio de espera entre a triagem e a primeira observação médica".

Foram identificadas 603 datas em falta para a ARS Algarve neste indicador, quando comparada com o conjunto de datas disponíveis nas restantes regiões. As falhas parecem estar espalhadas ao longo da série temporal, com maior concentração em alguns anos, nomeadamente 2017, 2019, 2024 e 2026.

Esta limitação será considerada nas análises regionais, sobretudo em comparações de tempo médio de espera e tendências temporais.

## 3. Preparação inicial dos dados

In [ ]:
df_eda = df_main.copy()

# Passar a coluna "Período" para formato datetime
df_eda["Período"] = pd.to_datetime(df_eda["Período"])

# Renomar colunas para facilitar o acesso
df_eda.rename(columns={
    "Período": "periodo",
    "Região/ARS": "regiao_ars",
    "Indicador": "indicador",
    "Valor": "valor",
    "Unidade": "unidade",
    "ID": "id"
}, inplace=True)

# Criar variáveis temporais
df_eda["ano"] = df_eda["periodo"].dt.year
df_eda["mes"] = df_eda["periodo"].dt.month
df_eda["ano_mes"] = df_eda["periodo"].dt.to_period("M")

# Identificar anos completos e ano parcial
df_eda["estado_ano"] = df_eda["ano"].apply(
    lambda x: "parcial" if x == 2026 else "completo"
)



df_eda.head()


Linhas antes: 82154
Linhas depois: 82154

Tipo da coluna periodo:
datetime64[us]

Anos disponíveis:
[np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]

Estado dos anos:
ano
2016    [completo]
2017    [completo]
2018    [completo]
2019    [completo]
2020    [completo]
2021    [completo]
2022    [completo]
2023    [completo]
2024    [completo]
2025    [completo]
2026     [parcial]
Name: estado_ano, dtype: object


,periodo,regiao_ars,indicador,valor,unidade,id,ano,mes,ano_mes,estado_ano
0,2017-02-13,ARS Lisboa e Vale do Tejo,Tempo médio de espera entre a triagem e a prim...,47.867240,minuto,2017-02-13/ARS Lisboa e Vale do Tejo/Tempo méd...,2017,2,2017-02,completo
1,2017-02-14,Portugal Continental,Tempo médio de espera entre a triagem e a prim...,46.503490,minuto,2017-02-14/Portugal Continental/Tempo médio de...,2017,2,2017-02,completo
2,2017-02-14,ARS Alentejo,Número estimado de episódios de urgência,1048.000000,episódio urgência,2017-02-14/ARS Alentejo/Número estimado de epi...,2017,2,2017-02,completo
3,2017-02-14,ARS Alentejo,Taxa diária de atendimentos urgentes com prior...,39.185390,epis. urg./100.000 resid.,2017-02-14/ARS Alentejo/Taxa diária de episódi...,2017,2,2017-02,completo
4,2017-02-14,ARS Norte,Taxa diária de atendimentos urgentes com inter...,8.190686,%,2017-02-14/ARS Norte/Taxa diária de episódios ...,2017,2,2017-02,completo


### Validação da preparação inicial

In [67]:
print("Linhas antes:", df_main.shape[0])
print("Linhas depois:", df_eda.shape[0])

print(f"\nTipo da coluna periodo: {df_eda['periodo'].dtype}")

print("\nAnos disponíveis:")
print(sorted(df_eda["ano"].unique()))

print("\nEstado dos anos:")
print(df_eda.groupby("ano")["estado_ano"].unique())

print(f"\nNúmero de valores nulos: {df_eda.isnull().sum().sum()}")


df_eda.head()

Linhas antes: 82154
Linhas depois: 82154

Tipo da coluna periodo: datetime64[us]

Anos disponíveis:
[np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]

Estado dos anos:
ano
2016    [completo]
2017    [completo]
2018    [completo]
2019    [completo]
2020    [completo]
2021    [completo]
2022    [completo]
2023    [completo]
2024    [completo]
2025    [completo]
2026     [parcial]
Name: estado_ano, dtype: object

Número de valores nulos: 0


,periodo,regiao_ars,indicador,valor,unidade,id,ano,mes,ano_mes,estado_ano
0,2017-02-13,ARS Lisboa e Vale do Tejo,Tempo médio de espera entre a triagem e a prim...,47.867240,minuto,2017-02-13/ARS Lisboa e Vale do Tejo/Tempo méd...,2017,2,2017-02,completo
1,2017-02-14,Portugal Continental,Tempo médio de espera entre a triagem e a prim...,46.503490,minuto,2017-02-14/Portugal Continental/Tempo médio de...,2017,2,2017-02,completo
2,2017-02-14,ARS Alentejo,Número estimado de episódios de urgência,1048.000000,episódio urgência,2017-02-14/ARS Alentejo/Número estimado de epi...,2017,2,2017-02,completo
3,2017-02-14,ARS Alentejo,Taxa diária de atendimentos urgentes com prior...,39.185390,epis. urg./100.000 resid.,2017-02-14/ARS Alentejo/Taxa diária de episódi...,2017,2,2017-02,completo
4,2017-02-14,ARS Norte,Taxa diária de atendimentos urgentes com inter...,8.190686,%,2017-02-14/ARS Norte/Taxa diária de episódios ...,2017,2,2017-02,completo


A preparação inicial dos dados manteve o número original de registos e confirmada a ausência de valores nulos. A coluna de período foi convertida para formato datetime e foram criadas variáveis temporais para apoiar a análise exploratória. O ano de 2026 foi identificado como ano parcial, uma vez que os dados disponíveis terminam em maio de 2026.

In [80]:
df_eda["indicador_curto"] = df_eda["indicador"].map({
    "Número estimado de episódios de urgência": "episodios_urgencia",
    "Taxa diária de atendimentos urgentes com prioridade verde ou azul": "prioridade_verde_azul",
    "Taxa diária de atendimentos urgentes com internamento": "taxa_internamento",
    "Tempo médio de espera entre a triagem e a primeira observação médica (rede de urgência hospitalar)": "tempo_medio_espera"
})

print(f"Ficaram {len(df_eda['indicador_curto'].unique())} indicadores curtos")

print(f"Ficaram {df_eda['indicador_curto'].isnull().sum()} valores em branco ou nulos")



Ficaram 4 indicadores curtos
Ficaram 0 valores em branco ou nulos


In [82]:
df_eda.groupby("indicador_curto")["unidade"].unique()

indicador_curto
episodios_urgencia               [episódio urgência]
prioridade_verde_azul    [epis. urg./100.000 resid.]
taxa_internamento                                [%]
tempo_medio_espera                          [minuto]
Name: unidade, dtype: object

In [83]:
df_eda.groupby("indicador_curto")["unidade"].nunique()

indicador_curto
episodios_urgencia       1
prioridade_verde_azul    1
taxa_internamento        1
tempo_medio_espera       1
Name: unidade, dtype: int64

In [84]:
df_eda.groupby(["indicador_curto", "unidade"]).size()

indicador_curto        unidade                  
episodios_urgencia     episódio urgência            20838
prioridade_verde_azul  epis. urg./100.000 resid.    20541
taxa_internamento      %                            20540
tempo_medio_espera     minuto                       20235
dtype: int64

### Criação de nomes curtos para os indicadores

Foi criada a coluna `indicador_curto` para facilitar a análise exploratória, mantendo a coluna `indicador` original para garantir rastreabilidade.

Cada indicador curto foi validado contra a respetiva unidade de medida.